# Installation and Import

- macOS: Run 'brew install poppler'
- Ubuntu/Debian: Run 'apt-get install poppler-utils'
- Windows: Install from https://github.com/oschwartz10612/poppler-windows/releases/ and add to PATH

In [ ]:
# !pip install typhoon-ocr==0.4.1
# !pip install python-dotenv
# !pip install lxml

In [ ]:
from dotenv import load_dotenv
import os
import pandas as pd
from typhoon_ocr import ocr_document
from io import StringIO
import re

load_dotenv()

# OCR

In [ ]:
API_KEY = os.getenv("TYPHOON_OCR_API_KEY")

In [ ]:
results = {}
files = [
    {
        "path": "data/district/advance/5-17 เขต 1 จ.บุรีรัมย์.pdf", 
        "pages": 16
    },
    {
        "path": "data/district/ontime/อำเภอบ้านด่าน/ตำบลบ้านด่าน.pdf",
        "pages": 40
    },
    {
        "path": "data/district/ontime/อำเภอบ้านด่าน/ตำบลปราสาท.pdf",
        "pages": 32
    },
    {
        "path": "data/district/ontime/อำเภอเมืองบุรีรัมย์/5ทับ18 ตำบลถลุงเหล็ก.pdf",
        "pages": 28
    },
    {
        "path": "data/district/ontime/อำเภอเมืองบุรีรัมย์/ตำบลกระสัง.pdf",
        "pages": 28
    },
    {
        "path": "data/district/ontime/อำเภอเมืองบุรีรัมย์/ตำบลกลันทา.pdf",
        "pages": 26
    },
    {
        "path": "data/district/ontime/อำเภอเมืองบุรีรัมย์/ตำบลชุมเห็ด.pdf",
        "pages": 64
    },
    {
        "path": "data/district/ontime/อำเภอเมืองบุรีรัมย์/ตำบลในเมือง.pdf",
        "pages": 62
    },
    {
        "path": "data/district/ontime/อำเภอเมืองบุรีรัมย์/ตำบลบัวทอง.pdf",
        "pages": 28
    },
    {
        "path": "data/district/ontime/อำเภอเมืองบุรีรัมย์/ตำบลบ้านบัว.pdf",
        "pages": 38
    },
    {
        "path": "data/district/ontime/อำเภอเมืองบุรีรัมย์/ตำบลบ้านยาง.pdf",
        "pages": 44
    },
    {
        "path": "data/district/ontime/อำเภอเมืองบุรีรัมย์/ตำบลพระครู.pdf",
        "pages": 22
    },
    {
        "path": "data\district\ontime\อำเภอเมืองบุรีรัมย์\ตำบลลุมปุ๊ก.pdf",
        "pages": 38,
    },
    {
        "path": "data/district/ontime/อำเภอเมืองบุรีรัมย์/ตำบลสะแกโพรง.pdf",
        "pages": 48,
    },
    {
        "path": "data/district/ontime/อำเภอเมืองบุรีรัมย์/ตำบลหนองตาด.pdf",
        "pages": 44
    }
    
]

In [ ]:
PARTICIPANTS_DISTRICT = [
    [
        "นายภาคภูมิ โภคทรัพย์", 
        "นายพีรภัทร ทองธีรสกุล", 
        "นายธนายุทธ ยืนยั่ง", 
        "นางสุทธิลักษณ์ ยายิรัมย์", 
        "นายสนอง เทพอักษรณรงค์", 
        "นายวิเชียร ลานทอง", 
        "นายนาท ฉัพพรรณธนกูร", 
        "นายอิทธิพัทธ์ ภักดีเนติพันธุ์"
    ],
    
    [
        "ประชาธิปัตย์", 
        "เพื่อไทย", 
        "ประชาชน", 
        "รวมไทยสร้างชาติ", 
        "ภูมิใจไทย", 
        "ประชากรไทย", 
        "กล้าธรรม", 
        "เศรษฐกิจ"
    ],
]

# อ่านข้อมูล file ตระกูลสส.เขต
def read_district(file_name, page_num):
    print(f"Processing District - File: {file_name} - Page: {page_num}")
    data_json = ocr_document(
        api_key=API_KEY,
        pdf_or_image_path=file_name,
        page_num=page_num,
        target_image_dim=2400,
    )
    
    good_card = re.search(r'บัตรดี\s+([\d,]+)\s*บัตร', data_json)
    bad_card = re.search(r'บัตรเสีย\s+([\d,]+)\s*บัตร', data_json)
    none_card = re.search(r'บัตรที่ไม่เลือกผู้สมัครผู้ใด\s+([\d,]+)\s*บัตร', data_json)
    
    summary = {
        'บัตรดี': int(good_card.group(1).replace(',', '')) if good_card else None,
        'บัตรเสีย': int(bad_card.group(1).replace(',', '')) if bad_card else None,
        'ไม่เลือกผู้ใด': int(none_card.group(1).replace(',', '')) if none_card else None,
    }
    
    df = pd.read_html(StringIO(data_json))[0]
        
    df.columns = ['หมายเลข', 'ชื่อผู้สมัคร', 'พรรค', 'คะแนน']
    df = df.iloc[1:].reset_index(drop=True)
    df.drop(columns=['หมายเลข'], inplace=True)
    
    # ตัดแถวให้เหลือแค่พรรคที่มี
    df = df.head(len(PARTICIPANTS_DISTRICT[0]))
    
    # ถ้าค่าใน column พรรค เป็นตัวเลข ให้เอามาใส่แทนที่ใน column คะแนน
    def swap_if_party_is_numeric(row):
        party_val = str(row['พรรค']).strip()
        if party_val.isdigit():
            return pd.Series({'พรรค': row['คะแนน'], 'คะแนน': party_val})
        return row
    df[['พรรค', 'คะแนน']] = df.apply(swap_if_party_is_numeric, axis=1)[['พรรค', 'คะแนน']]
    
    # เติมชื่อผู้สมัครและพรรค
    df["ชื่อผู้สมัคร"] = PARTICIPANTS_DISTRICT[0]
    df["พรรค"] = PARTICIPANTS_DISTRICT[1]

    # Extract เฉพาะตัวเลขจากคะแนน
    df['คะแนน'] = df['คะแนน'].astype(str).str.extract(r'(\d+)')[0].astype(int)
    
    # เช็คความถูกต้องของคะแนนรวมกับบัตรดี
    votes_sum = df['คะแนน'].sum()
    good_cards = summary['บัตรดี']

    validation = {
        'num_good_cards': int(good_cards) if good_cards else None,
        'total_scores': int(votes_sum) if votes_sum else None,
    }

    print(f"Validation: {validation}")
    return summary, df, validation

### Read all PDF Files

In [ ]:
for file_config in files:
    if "district" in file_config["path"]:
        total_pages = file_config["pages"]
        for page_num in range(1, total_pages, 2):
            summary, df, validation = read_district(file_config["path"], page_num)
            results[(file_config["path"], page_num)] = (summary, df, validation)
            print(f"Summary for {file_config['path']} - Page {page_num}: {summary}")
            print(df)
            print(f"Validation for {file_config['path']} - Page {page_num}: {validation}")
            print("-" * 50)
            